In [ ]:
!wget "https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/pub.pt" \
"https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/priv.pt" \
"https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/model.pt" \
"https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/task_template.py"

In [ ]:
# config
BASE = "/kaggle/working/"
PUB_PATH = BASE + "pub.pt"
PRIV_PATH = BASE + "priv.pt"
MODEL_PATH = BASE + "model.pt"
OUTPUT_CSV = BASE + "submission.csv"

BASE_URL = "http://34.63.153.158"   #DONOT CHANGE
API_KEY = "YOUR_API_KEY_HERE"
TASK_ID = "01-mia"  #DONOT CHANGE



In [ ]:
import os
import sys
import torch
import pandas as pd
import requests
import random
from tqdm import tqdm

from pathlib import Path
from torch.utils.data import Dataset
from torchvision.models import resnet18
import torchvision.transforms as transforms

import torch.nn as nn
from torchvision.models import resnet18
from torch.utils.data import DataLoader, Subset
import numpy as np
from tqdm import tqdm

# dataset classes
class TaskDataset(Dataset):
    def __init__(self, transform=None):
        self.ids = []
        self.imgs = []
        self.labels = []
        self.transform = transform

    def __getitem__(self, index):
        id_ = self.ids[index]
        img = self.imgs[index]
        if self.transform is not None:
            img = self.transform(img)
        label = self.labels[index]
        return id_, img, label

    def __len__(self):
        return len(self.ids)


class MembershipDataset(TaskDataset):
    def __init__(self, transform=None):
        super().__init__(transform)
        self.membership = []

    def __getitem__(self, index):
        id_, img, label = super().__getitem__(index)
        return id_, img, label, self.membership[index]


# load datasets
print("Loading datasets...")
pub_ds = torch.load(PUB_PATH, weights_only=False)
priv_ds = torch.load(PRIV_PATH, weights_only=False)

pub_size  = len(pub_ds)
priv_size = len(priv_ds)

# normalization (same as training)
MEAN = [0.7406, 0.5331, 0.7059]
STD = [0.1491, 0.1864, 0.1301]

transform = transforms.Compose([
    transforms.Resize(32),
    transforms.Normalize(mean=MEAN, std=STD),
])

pub_ds.transform = transform
priv_ds.transform = transform

In [ ]:
# load model
print("Loading model...")
model = resnet18(weights=None)
model.conv1 = torch.nn.Conv2d(3, 64, 3, 1, 1, bias=False)
model.maxpool = torch.nn.Identity()
model.fc = torch.nn.Linear(512, 9)

model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

# add this after loading the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")

In [ ]:

# ── helpers ──────────────────────────────────────────────────────────────────

def get_model():
    m = resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Linear(512, 9)
    return m.to(device)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    for _, img, label, _ in loader:
        img, label = img.to(device), label.to(device)
        optimizer.zero_grad()
        loss = criterion(model(img), label)
        loss.backward()
        optimizer.step()


def collate_ignore_none(batch):
    # batch is a list of tuples (id, img, label, membership)
    ids    = [item[0] for item in batch]
    imgs   = torch.stack([item[1] for item in batch])
    labels = torch.tensor([item[2] for item in batch])
    return ids, imgs, labels

def get_logit_confidences(model, dataset):
    """Run model on dataset, return logit-scaled true-label confidences."""
    model.eval()
    confs = []
    
    loader = DataLoader(
        dataset, 
        batch_size=256, 
        shuffle=False, 
        num_workers=0,
        collate_fn=collate_ignore_none  # handles None membership
    )
    
    with torch.no_grad():
        for id_, img, label in loader:
            img, label = img.to(device), label.to(device)
            probs = torch.softmax(model(img), dim=1)
            conf = probs[torch.arange(len(label)), label].cpu().numpy()
            logit_conf = np.log(conf / (1 - conf + 1e-10))
            confs.extend(logit_conf)
    
    return np.array(confs)


In [ ]:
# ── config ────────────────────────────────────────────────────────────────────

N_SHADOW = 64
N_EPOCHS = 50
LR       = 0.05
SAVE_DIR = Path("shadow_models")
SAVE_DIR.mkdir(exist_ok=True)

pub_size  = len(pub_ds)
priv_size = len(priv_ds)

# ── train shadow models ───────────────────────────────────────────────────────

shadow_results = []   # list of dicts, one per shadow model

for idx in range(N_SHADOW):
    print(f"\n{'='*50}")
    print(f"Shadow model {idx+1} / {N_SHADOW}")
    print(f"{'='*50}")

    # --- random 50/50 split of pub.pt ---
    in_mask     = np.random.rand(pub_size) < 0.5
    in_indices  = np.where(in_mask)[0]

    train_loader = DataLoader(
        Subset(pub_ds, in_indices),
        batch_size=256, shuffle=True, num_workers=2
    )

    # --- train ---
    shadow = get_model()
    optimizer = torch.optim.SGD(shadow.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    criterion = nn.CrossEntropyLoss()

    for epoch in tqdm(range(N_EPOCHS), desc=f"Training shadow {idx+1}"):
        train_one_epoch(shadow, train_loader, optimizer, criterion)
        scheduler.step()

    # --- save model weights ---
    model_path = SAVE_DIR / f"shadow_{idx+1}.pt"
    torch.save(shadow.state_dict(), model_path)
    print(f"Saved → {model_path}")

    # --- run inference on pub.pt and priv.pt ---
    print("Running inference on pub.pt ...")
    pub_confs  = get_logit_confidences(shadow, pub_ds)

    print("Running inference on priv.pt ...")
    priv_confs = get_logit_confidences(shadow, priv_ds)

    # --- store everything we need later ---
    shadow_results.append({
        "in_mask"   : in_mask,       # bool array [pub_size]  — was sample in training?
        "pub_confs" : pub_confs,      # float array [pub_size]
        "priv_confs": priv_confs,     # float array [priv_size]
    })

    print(f"Shadow {idx+1} done ✓")

print("\nAll shadow models trained and saved!")

In [ ]:
from scipy.stats import norm
from sklearn.metrics import roc_curve

# ── target model confidences on pub.pt ───────────────────────────────────────
all_confs = []
all_memberships = []

with torch.no_grad():
    for id_, img, label, membership in tqdm(DataLoader(pub_ds, batch_size=256, shuffle=False, num_workers=2), desc="Target model on pub.pt"):
        img = img.to(device)
        label = label.to(device)
        probs = torch.softmax(model(img), dim=1)
        conf = probs[torch.arange(len(label)), label]
        all_confs.extend(conf.cpu().numpy())
        all_memberships.extend(membership.numpy())

all_confs = np.array(all_confs)
all_memberships = np.array(all_memberships)


# ── LiRA scoring on pub.pt (we know ground truth here) ───────────────────────

target_logit_confs_pub = np.log(all_confs / (1 - all_confs + 1e-10))

lira_scores_pub = []

for i in tqdm(range(pub_size), desc="LiRA scoring pub.pt"):
    in_confs  = []
    out_confs = []
    
    for result in shadow_results:
        if result["in_mask"][i]:
            in_confs.append(result["pub_confs"][i])
        else:
            out_confs.append(result["pub_confs"][i])
    
    # fallback if one side is empty (likely with only 1 shadow model)
    if len(in_confs) < 1 or len(out_confs) < 1:
        lira_scores_pub.append(target_logit_confs_pub[i])
        continue
    
    mu_out    = np.mean(out_confs)
    sigma_out = np.std(out_confs) + 1e-8
    
    mu_in     = np.mean(in_confs)
    sigma_in  = np.std(in_confs) + 1e-8
    
    target_conf = target_logit_confs_pub[i]
    log_prob_in  = norm.logpdf(target_conf, mu_in,  sigma_in)
    log_prob_out = norm.logpdf(target_conf, mu_out, sigma_out)
    
    score = log_prob_in - log_prob_out
    lira_scores_pub.append(score)

lira_scores_pub = np.array(lira_scores_pub)

# evaluate
fpr, tpr, _ = roc_curve(all_memberships, lira_scores_pub)
tpr_at_5fpr  = tpr[np.searchsorted(fpr, 0.05)]
print(f"LiRA TPR@5%FPR on pub.pt: {tpr_at_5fpr:.4f}")

In [ ]:
from scipy.stats import norm

# ── step 1: target model confidences on priv.pt ──────────────────────────────
print("Step 1: Target model inference on priv.pt...")

all_priv_confs = []

with torch.no_grad():
    for batch in tqdm(DataLoader(priv_ds, batch_size=256, shuffle=False,
                                  num_workers=0, collate_fn=collate_ignore_none),
                      desc="Target model → priv.pt"):
        _, img, label = batch
        img, label = img.to(device), label.to(device)
        probs = torch.softmax(model(img), dim=1)
        conf  = probs[torch.arange(len(label)), label].cpu().numpy()
        all_priv_confs.extend(conf)

all_priv_confs   = np.array(all_priv_confs)
priv_logit_confs = np.log(all_priv_confs / (1 - all_priv_confs + 1e-10))
print(f"  priv logit confs shape: {priv_logit_confs.shape}")

# ── step 2: LiRA scores for priv.pt ──────────────────────────────────────────
print("\nStep 2: Computing LiRA scores for priv.pt...")

lira_scores_priv = []

for i in tqdm(range(priv_size), desc="LiRA scoring priv.pt"):

    # all shadow confidences on priv sample i = out distribution
    # (priv samples were never in any shadow training)
    out_confs = [result["priv_confs"][i] for result in shadow_results]

    mu_out    = np.mean(out_confs)
    sigma_out = np.std(out_confs) + 1e-8

    # how far above the out distribution is the target model's confidence?
    target_conf = priv_logit_confs[i]
    score       = (target_conf - mu_out) / sigma_out

    lira_scores_priv.append(score)

lira_scores_priv = np.array(lira_scores_priv)

# ── step 3: normalize to [0, 1] ──────────────────────────────────────────────
print("\nStep 3: Normalizing scores...")

scores = (lira_scores_priv - lira_scores_priv.min()) / (lira_scores_priv.max() - lira_scores_priv.min())

# ── step 4: write csv ─────────────────────────────────────────────────────────
print("\nStep 4: Writing submission.csv...")

df = pd.DataFrame({
    "id"   : [str(i) for i in priv_ds.ids],
    "score": scores
})

df.to_csv(OUTPUT_CSV, index=False)
print(f"  Saved {len(df)} rows to {OUTPUT_CSV}")
print(df.head())

In [ ]:
# submit
def die(msg):
    print(msg, file=sys.stderr)
    sys.exit(1)

# parser = argparse.ArgumentParser(description="Submit a CSV file to the server.")
# args = parser.parse_args()

submit_path = Path(OUTPUT_CSV)

if not submit_path.exists():
    die(f"File not found: {submit_path}")

try:
    with open(submit_path, "rb") as f:
        resp = requests.post(
            f"{BASE_URL}/submit/{TASK_ID}",
            headers={"X-API-Key": API_KEY},
            files={"file": (submit_path.name, f, "application/csv")},
            timeout=(10, 600),
        )
    try:
        body = resp.json()
    except Exception:
        body = {"raw_text": resp.text}

    if resp.status_code == 413:
        die("Upload rejected: file too large (HTTP 413).")

    resp.raise_for_status()

    print("Successfully submitted.")
    print("Server response:", body)
    submission_id = body.get("submission_id")
    if submission_id:
        print(f"Submission ID: {submission_id}")

except requests.exceptions.RequestException as e:
    detail = getattr(e, "response", None)
    print(f"Submission error: {e}")
    if detail is not None:
        try:
            print("Server response:", detail.json())
        except Exception:
            print("Server response (text):", detail.text)
    sys.exit(1)